In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q rfdetr==1.3.0 supervision==0.26.1 roboflow==1.2.10

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY") )
project = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-3-ycjdo")
version = project.version(10)
dataset = version.download("coco")


In [ ]:
from rfdetr import RFDETRSegPreview

model = RFDETRSegPreview(segmentation_head=False)

model.train(dataset_dir=dataset.location, epochs=5, batch_size=4, grad_accum_steps=4, segmentation_head=False)

In [ ]:
from PIL import Image

Image.open("/content/output/metrics_plot.png")

In [ ]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [ ]:
cleanup_gpu_memory(model, verbose=True)

In [ ]:
model = RFDETRSegPreview(pretrain_weights="/content/output/checkpoint_best_total.pth")
model.optimize_for_inference()

In [ ]:
import supervision as sv

ds_test = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
    force_masks=True                                    
                                                                
                                                                )